In [1]:
from pathlib import Path
import pandas as pd

# Load every saved raw season file
raw_files = sorted(Path("data/raw").glob("epl_*.csv"))

matches = pd.concat(
    [pd.read_csv(file, encoding="latin1") for file in raw_files],
    ignore_index=True
)

# Keep only information known from a completed match
keep_columns = [
    "Season", "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR"
]

clean_matches = matches[keep_columns].copy()

print(f"Rows loaded: {len(clean_matches)}")
print("\nMissing values:")
print(clean_matches.isna().sum())

print("\nFirst five matches:")
display(clean_matches.head())

Rows loaded: 3800

Missing values:
Season      0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
dtype: int64

First five matches:


,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,2015-16,08/08/2015,Bournemouth,Aston Villa,0,1,A
1,2015-16,08/08/2015,Chelsea,Swansea,2,2,D
2,2015-16,08/08/2015,Everton,Watford,2,2,D
3,2015-16,08/08/2015,Leicester,Sunderland,4,2,H
4,2015-16,08/08/2015,Man United,Tottenham,1,0,H


In [2]:
print(clean_matches.isna().sum().to_string())

Season      0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0


In [3]:
# Convert the date text into real dates
clean_matches["Date"] = pd.to_datetime(
    clean_matches["Date"],
    dayfirst=True,
    errors="coerce"
)

# Make goal columns numeric whole numbers
clean_matches["FTHG"] = pd.to_numeric(clean_matches["FTHG"], errors="coerce")
clean_matches["FTAG"] = pd.to_numeric(clean_matches["FTAG"], errors="coerce")

# Remove rows that cannot be used for modelling
before_cleaning = len(clean_matches)

clean_matches = clean_matches.dropna(
    subset=["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]
)
clean_matches = clean_matches.drop_duplicates()

# Store goals as integers and sort oldest to newest
clean_matches["FTHG"] = clean_matches["FTHG"].astype(int)
clean_matches["FTAG"] = clean_matches["FTAG"].astype(int)
clean_matches = clean_matches.sort_values("Date").reset_index(drop=True)

# Save the clean version
clean_file = Path("data/processed/epl_matches_clean.csv")
clean_matches.to_csv(clean_file, index=False)

print(f"Rows before cleaning: {before_cleaning}")
print(f"Rows after cleaning:  {len(clean_matches)}")
print(f"Saved clean data to:   {clean_file}")
display(clean_matches.head())

Rows before cleaning: 3800
Rows after cleaning:  3420
Saved clean data to:   data\processed\epl_matches_clean.csv


,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,2015-16,2015-08-08,Bournemouth,Aston Villa,0,1,A
1,2015-16,2015-08-08,Chelsea,Swansea,2,2,D
2,2015-16,2015-08-08,Everton,Watford,2,2,D
3,2015-16,2015-08-08,Leicester,Sunderland,4,2,H
4,2015-16,2015-08-08,Man United,Tottenham,1,0,H


In [4]:
print("Clean matches ready:", len(clean_matches))

Clean matches ready: 3420


In [6]:
# Rebuild the clean table from the untouched downloaded data
clean_matches = matches[keep_columns].copy()

# "mixed" safely handles date-format variations between seasons
clean_matches["Date"] = pd.to_datetime(
    clean_matches["Date"],
    dayfirst=True,
    format="mixed",
    errors="coerce"
)

clean_matches["FTHG"] = pd.to_numeric(clean_matches["FTHG"], errors="coerce")
clean_matches["FTAG"] = pd.to_numeric(clean_matches["FTAG"], errors="coerce")

clean_matches = clean_matches.dropna(
    subset=["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]
).drop_duplicates()

clean_matches["FTHG"] = clean_matches["FTHG"].astype(int)
clean_matches["FTAG"] = clean_matches["FTAG"].astype(int)

clean_matches = clean_matches.sort_values("Date").reset_index(drop=True)
clean_matches.to_csv("data/processed/epl_matches_clean.csv", index=False)

print("Clean matches ready:", len(clean_matches))
print(clean_matches.groupby("Season").size())

Clean matches ready: 3800
Season
2015-16    380
2016-17    380
2017-18    380
2018-19    380
2019-20    380
2020-21    380
2021-22    380
2022-23    380
2023-24    380
2024-25    380
dtype: int64


In [7]:
from pathlib import Path

raw_files = sorted(Path("data/raw").glob("epl_*.csv"))

print("Files found:")
for file in raw_files:
    print(file.name)

Files found:
epl_1516.csv
epl_1617.csv
epl_1718.csv
epl_1819.csv
epl_1920.csv
epl_2021.csv
epl_2122.csv
epl_2223.csv
epl_2324.csv
epl_2425.csv
epl_2526.csv
epl_2627.csv


In [8]:
import pandas as pd
from pathlib import Path

# Reload every raw season file from disk
raw_files = sorted(Path("data/raw").glob("epl_*.csv"))

matches = pd.concat(
    [pd.read_csv(file, encoding="latin1") for file in raw_files],
    ignore_index=True
)

keep_columns = [
    "Season", "Date", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR"
]

clean_matches = matches[keep_columns].copy()

clean_matches["Date"] = pd.to_datetime(
    clean_matches["Date"],
    dayfirst=True,
    format="mixed",
    errors="coerce"
)

clean_matches["FTHG"] = pd.to_numeric(clean_matches["FTHG"], errors="coerce")
clean_matches["FTAG"] = pd.to_numeric(clean_matches["FTAG"], errors="coerce")

# This removes unfinished 2026–27 fixtures with no score yet
clean_matches = clean_matches.dropna(
    subset=["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"]
).drop_duplicates()

clean_matches["FTHG"] = clean_matches["FTHG"].astype(int)
clean_matches["FTAG"] = clean_matches["FTAG"].astype(int)

clean_matches = clean_matches.sort_values("Date").reset_index(drop=True)
clean_matches.to_csv("data/processed/epl_matches_clean.csv", index=False)

print("Clean matches ready:", len(clean_matches))
print("Latest match:", clean_matches["Date"].max().date())

Clean matches ready: 4230
Latest match: 2026-09-20
